In [1]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, Dataset

In [2]:
# 定义自定义数据集类
class MyDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data) - 21

    def __getitem__(self, idx):
        # 返回前20行作为inputs，第21行作为label
        return self.data[idx:idx+20], self.data[idx+20]

In [3]:
# 检查GPU可用性
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [4]:
# 定义LSTM模型
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, input_size)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])
        return out

In [5]:
# 定义训练函数
def train(model, dataloader, sequence_length, epochs, learning_rate):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    model.to(device)  # 将模型移动到GPU

    for epoch in range(epochs):
        for inputs,label in dataloader:
#             print(inputs.shape,label.shape)
            inputs = inputs.float().to(device)
            label = label.float().to(device)
#             inputs = inputs.unsqueeze(0).float().to(device)
#             label = label.unsqueeze(0).float().to(device)

            # 前向传播和反向传播
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, label)
            loss.backward()
            optimizer.step()

        # 输出当前轮次的损失值
        print(f'Epoch: {epoch+1}/{epochs}, Loss: {loss.item()}')

In [6]:
# 创建模型和数据集
input_size = 1538
hidden_size = 2000
num_layers = 2
sequence_length = 20
epochs = 200
learning_rate = 0.001
model = LSTMModel(input_size, hidden_size, num_layers)

In [7]:
# 读取数据集
file_dir = 'E:/dataset/aruba1'
dataset_num = 0
dataset = np.load(f'{file_dir}/dataset{dataset_num}.npy')
print(dataset.shape)
dataset = dataset[:10000]
dataset.shape

(200000, 1538)


(10000, 1538)

In [8]:
my_dataset = MyDataset(dataset)

In [9]:
my_dataset[0]

(array([[ 0.84110356, -1.30194879, -0.11225183, ..., -0.35486972,
         -0.19016711,  1.        ],
        [ 0.84110378, -1.30194879, -0.11225183, ..., -0.35486972,
         -0.19016711,  0.        ],
        [ 0.84112509, -0.85931224, -0.27637973, ..., -0.20529866,
         -0.16735311, 21.5       ],
        ...,
        [ 0.84140401, -0.85095537, -0.14457437, ..., -0.20529866,
         -0.16735311, 20.        ],
        [ 0.84143289, -0.87211657, -0.13763657, ..., -0.20529866,
         -0.16735311, 19.5       ],
        [ 0.84148097, -0.81916559, -0.12867293, ..., -0.20529866,
         -0.16735311, 20.        ]]),
 array([ 0.84151934, -1.30194879, -0.11225183, ..., -0.35486972,
        -0.19016711,  1.        ]))

In [10]:
dataloader = DataLoader(my_dataset, batch_size=1, shuffle=True)

In [11]:
train(model, dataloader, sequence_length, epochs, learning_rate)

Epoch: 1/200, Loss: 0.01524145808070898
Epoch: 2/200, Loss: 0.020509134978055954
Epoch: 3/200, Loss: 0.009192291647195816
Epoch: 4/200, Loss: 0.020404987037181854
Epoch: 5/200, Loss: 0.012032199651002884
Epoch: 6/200, Loss: 0.015679048374295235
Epoch: 7/200, Loss: 0.010805034078657627
Epoch: 8/200, Loss: 0.02542280964553356
Epoch: 9/200, Loss: 0.03075142204761505



KeyboardInterrupt



In [ ]:
n = 0  # 替换为想要预测的序列起始位置
input_sequence = dataset[n:n+sequence_length]
input_sequence = torch.tensor(input_sequence).unsqueeze(0).float().to(device)
prediction = model(input_sequence)
# print(f'Prediction: {prediction.squeeze().tolist()}')
prediction.squeeze().tolist()[0]